# 08 — JAX vs PyTorch Benchmarks

We measure wall-clock time per training step for both frameworks across different model sizes. Results are written to `benchmarks/results/` by the CLI script `benchmarks/run_benchmarks.py`, then plotted here.

## What we benchmark
- **Training step**: forward + backward + optimizer update
- **Variables swept**: `batch_size`, `seq_len`, `hidden_size`, `rank` (for low-rank)
- **Metric**: median ms/step over 20 steps after 5 warmup steps

## Important caveat on JAX timing
JAX operations are **asynchronous by default**. Always call `.block_until_ready()` before stopping the timer.

In [ ]:
import sys; sys.path.insert(0, '..')
import subprocess, json, time
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

RESULTS_DIR = Path('../benchmarks/results')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

## 8.1 Run Benchmarks via CLI Script

Uncomment the cells below to run benchmarks. Each call writes a JSON file to `benchmarks/results/`.

In [ ]:
# Quick benchmark across hidden sizes
hidden_sizes = [32, 64, 128, 256]
for h in hidden_sizes:
    result = subprocess.run(
        [sys.executable, '../benchmarks/run_benchmarks.py',
         '--framework', 'both',
         '--model', 'vanilla',
         '--hidden', str(h),
         '--seq-len', '100',
         '--batch', '64'],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print(f'hidden={h}: ERROR\n', result.stderr[:300])
    else:
        print(result.stdout.strip())

In [ ]:
# Benchmark across sequence lengths
seq_lens = [50, 100, 200, 500]
for s in seq_lens:
    result = subprocess.run(
        [sys.executable, '../benchmarks/run_benchmarks.py',
         '--framework', 'both',
         '--model', 'vanilla',
         '--hidden', '64',
         '--seq-len', str(s),
         '--batch', '64'],
        capture_output=True, text=True
    )
    print(result.stdout.strip())

In [ ]:
# Low-rank: sweep over rank values
for r in [1, 2, 4, 8, 16]:
    result = subprocess.run(
        [sys.executable, '../benchmarks/run_benchmarks.py',
         '--framework', 'both',
         '--model', 'lowrank',
         '--hidden', '64',
         '--seq-len', '100',
         '--batch', '64',
         '--rank', str(r)],
        capture_output=True, text=True
    )
    print(result.stdout.strip())

## 8.2 Load and Plot Results

In [ ]:
def load_results(results_dir: Path) -> list[dict]:
    results = []
    for f in sorted(results_dir.glob('*.json')):
        with open(f) as fp:
            results.append(json.load(fp))
    return results

results = load_results(RESULTS_DIR)
print(f'Loaded {len(results)} result files')
if results:
    print('Example entry keys:', list(results[0].keys()))

In [ ]:
def plot_comparison(results, x_key, title, xlabel):
    """Plot JAX vs PyTorch median step time, sweeping over x_key."""
    jax_pts   = sorted([(r['config'][x_key], r['median_ms']) for r in results if r['framework']=='jax'])
    torch_pts = sorted([(r['config'][x_key], r['median_ms']) for r in results if r['framework']=='torch'])

    if not jax_pts or not torch_pts:
        print(f'Not enough data for {x_key} sweep — run the benchmark cells above first.')
        return

    xs_jax,   ys_jax   = zip(*jax_pts)
    xs_torch, ys_torch = zip(*torch_pts)

    plt.figure(figsize=(7, 4))
    plt.plot(xs_jax, ys_jax, 'o-', label='JAX (jit+scan)')
    plt.plot(xs_torch, ys_torch, 's--', label='PyTorch')
    plt.xlabel(xlabel); plt.ylabel('Median step time (ms)')
    plt.title(title)
    plt.legend(); plt.grid(alpha=0.3); plt.show()

# Filter results by sweep type and plot
vanilla_results = [r for r in results if r['config'].get('model') == 'vanilla']
hidden_sweep    = [r for r in vanilla_results if r['config'].get('seq_len') == 100]
seqlen_sweep    = [r for r in vanilla_results if r['config'].get('hidden') == 64]

plot_comparison(hidden_sweep,  'hidden', 'Training step time vs hidden size (T=100, B=64)', 'Hidden size')
plot_comparison(seqlen_sweep, 'seq_len', 'Training step time vs sequence length (H=64, B=64)', 'Sequence length')

## 8.3 Compile Time vs Runtime

JAX pays a one-time compilation cost on the first call. After that, subsequent calls are much faster. This section measures both.

In [ ]:
import jax
import jax.numpy as jnp
import flax.nnx as nnx
import optax
import torch
from src.rnn_jax import VanillaRNNModel
from src.rnn_torch import VanillaRNNModel as TorchModel
from src.train import mse_loss, make_train_step

H, T_LEN, B = 64, 100, 64

# --- JAX — batch-first (B, T, I) ---
jax_model = VanillaRNNModel(1, H, 1, rngs=nnx.Rngs(0))
jax_opt   = nnx.Optimizer(jax_model, optax.adam(1e-3), wrt=nnx.Param)

def jax_loss(model, batch): return mse_loss(model(batch['inputs']), batch['targets'])
jax_train = make_train_step(jax_loss)

key = jax.random.PRNGKey(0)
xs_jax = jax.random.normal(key, (B, T_LEN, 1))   # (B, T, I)
batch_jax = {'inputs': xs_jax, 'targets': xs_jax}

t_compile_start = time.perf_counter()
_ = jax_train(jax_model, jax_opt, batch_jax)  # triggers compilation
jax.effects_barrier()
t_compile = (time.perf_counter() - t_compile_start) * 1000

times_jax = []
for _ in range(20):
    t0 = time.perf_counter()
    jax_train(jax_model, jax_opt, batch_jax)
    jax.effects_barrier()
    times_jax.append((time.perf_counter()-t0)*1000)

# --- PyTorch — batch-first (B, T, I) ---
torch_model = TorchModel(1, H, 1)
torch_opt   = torch.optim.Adam(torch_model.parameters(), lr=1e-3)
xs_torch    = torch.randn(B, T_LEN, 1)   # (B, T, I)

times_torch = []
for _ in range(25):
    t0 = time.perf_counter()
    torch_opt.zero_grad()
    out = torch_model(xs_torch)
    loss = (out - xs_torch).pow(2).mean()
    loss.backward()
    torch_opt.step()
    times_torch.append((time.perf_counter()-t0)*1000)

print(f'JAX compile time:     {t_compile:.0f} ms (one-time)')
print(f'JAX  median step:     {np.median(times_jax):.2f} ms')
print(f'PyTorch median step:  {np.median(times_torch):.2f} ms')

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(times_jax, 'o-', label='JAX (post-compile)')
ax.plot(times_torch, 's--', label='PyTorch')
ax.set(xlabel='Step', ylabel='Wall time (ms)', title=f'Step time (H={H}, T={T_LEN}, B={B})')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()